# Exercise: Binning and Pivot Tables

This notebook gives you practice applying the tools from `02_combining_dataframes.ipynb`: joining and concatenating DataFrames with `.join()` and `pd.concat()`, binning continuous data into categories with `pd.cut()`, and summarising data with `pd.pivot_table()`. You will work with the red and white wine quality datasets throughout, loading, combining, filtering, and pivoting them to answer realistic data questions.

---

## Starter Code

In [1]:
# Import pandas and load both wine datasets, run this cell first
import pandas as pd

red_df = pd.read_csv("../data/winequality-red.csv", delimiter=";")
white_df = pd.read_csv("../data/winequality-white.csv", delimiter=";")

print("Red wine shape:", red_df.shape)
print("White wine shape:", white_df.shape)

Red wine shape: (1599, 12)
White wine shape: (4898, 12)


---

## Part 1: Loading and Inspecting

### Challenge 1: Sense-check the data

Use DataFrame attributes and methods to confirm both datasets loaded correctly. Check:

- The number of rows and columns in each

- The data types of the columns

- Whether any columns contain null values

In [2]:
# Check shape, dtypes, and null counts for both datasets
for wine_name, wine_data in [("Red", red_df), ("White", white_df)]:
    print(f"{wine_name} wine shape: {wine_data.shape}")
    print(f"{wine_name} wine dtypes:")
    display(wine_data.dtypes.rename("dtype").to_frame())
    print(f"{wine_name} wine null counts:")
    display(wine_data.isna().sum().rename("null_count").to_frame())

Red wine shape: (1599, 12)
Red wine dtypes:


,dtype
fixed acidity,float64
volatile acidity,float64
citric acid,float64
residual sugar,float64
chlorides,float64
free sulfur dioxide,float64
total sulfur dioxide,float64
density,float64
pH,float64
sulphates,float64


Red wine null counts:


,null_count
fixed acidity,0
volatile acidity,0
citric acid,0
residual sugar,0
chlorides,0
free sulfur dioxide,0
total sulfur dioxide,0
density,0
pH,0
sulphates,0


White wine shape: (4898, 12)
White wine dtypes:


,dtype
fixed acidity,float64
volatile acidity,float64
citric acid,float64
residual sugar,float64
chlorides,float64
free sulfur dioxide,float64
total sulfur dioxide,float64
density,float64
pH,float64
sulphates,float64


White wine null counts:


,null_count
fixed acidity,0
volatile acidity,0
citric acid,0
residual sugar,0
chlorides,0
free sulfur dioxide,0
total sulfur dioxide,0
density,0
pH,0
sulphates,0


---

## Part 2: Filtering by Alcohol Content

### Challenge 2: Flag above-average alcohol

You want to avoid wines with above-average alcohol content. To do this:

1. Find the **mean alcohol content** separately for reds and whites.

2. Create a **boolean `Series`** for each DataFrame indicating whether each row has an alcohol content above the mean.

3. Attach this boolean `Series` to each DataFrame, do it **twice**: once using `.join()` and once using `pd.concat(..., axis=1)`.

4. Filter each DataFrame to return only the rows where alcohol is **above** the mean.

In [3]:
# Mean alcohol content by wine type
red_mean_alcohol = red_df["alcohol"].mean()
white_mean_alcohol = white_df["alcohol"].mean()
print(f"Red mean alcohol: {red_mean_alcohol:.3f}")
print(f"White mean alcohol: {white_mean_alcohol:.3f}")

Red mean alcohol: 10.423
White mean alcohol: 10.514


In [4]:
# Boolean Series: True where alcohol exceeds that wine type's mean
red_above_mean = (red_df["alcohol"] > red_mean_alcohol).rename("above_mean_alcohol")
white_above_mean = (white_df["alcohol"] > white_mean_alcohol).rename("above_mean_alcohol")
print("Red wines above mean:", int(red_above_mean.sum()))
print("White wines above mean:", int(white_above_mean.sum()))

Red wines above mean: 683
White wines above mean: 2163


In [5]:
# Attach boolean flags with DataFrame.join(), aligned by index
red_alcohol_joined = red_df.join(red_above_mean)
white_alcohol_joined = white_df.join(white_above_mean)
red_alcohol_joined.head(), white_alcohol_joined.head()

(   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
 0            7.4              0.70         0.00             1.9      0.076   
 1            7.8              0.88         0.00             2.6      0.098   
 2            7.8              0.76         0.04             2.3      0.092   
 3           11.2              0.28         0.56             1.9      0.075   
 4            7.4              0.70         0.00             1.9      0.076   
 
    free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
 0                 11.0                  34.0   0.9978  3.51       0.56   
 1                 25.0                  67.0   0.9968  3.20       0.68   
 2                 15.0                  54.0   0.9970  3.26       0.65   
 3                 17.0                  60.0   0.9980  3.16       0.58   
 4                 11.0                  34.0   0.9978  3.51       0.56   
 
    alcohol  quality  above_mean_alcohol  
 0      9.4        5         

In [6]:
# Attach the same boolean flags with pd.concat() along columns
red_alcohol_concat = pd.concat([red_df, red_above_mean], axis=1)
white_alcohol_concat = pd.concat([white_df, white_above_mean], axis=1)
red_alcohol_concat.head(), white_alcohol_concat.head()

(   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
 0            7.4              0.70         0.00             1.9      0.076   
 1            7.8              0.88         0.00             2.6      0.098   
 2            7.8              0.76         0.04             2.3      0.092   
 3           11.2              0.28         0.56             1.9      0.075   
 4            7.4              0.70         0.00             1.9      0.076   
 
    free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
 0                 11.0                  34.0   0.9978  3.51       0.56   
 1                 25.0                  67.0   0.9968  3.20       0.68   
 2                 15.0                  54.0   0.9970  3.26       0.65   
 3                 17.0                  60.0   0.9980  3.16       0.58   
 4                 11.0                  34.0   0.9978  3.51       0.56   
 
    alcohol  quality  above_mean_alcohol  
 0      9.4        5         

In [7]:
# Keep only wines whose alcohol exceeds the corresponding dataset mean
red_above_mean_wines = red_alcohol_joined.loc[red_alcohol_joined["above_mean_alcohol"]].copy()
white_above_mean_wines = white_alcohol_joined.loc[white_alcohol_joined["above_mean_alcohol"]].copy()
print("Red wines above mean:", red_above_mean_wines.shape)
print("White wines above mean:", white_above_mean_wines.shape)
display(red_above_mean_wines.head())
display(white_above_mean_wines.head())

Red wines above mean: (683, 13)
White wines above mean: (2163, 13)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,above_mean_alcohol
9,7.5,0.500,0.36,6.1,0.071,17.0,102.0,0.9978,3.35,0.80,10.5,5,True
11,7.5,0.500,0.36,6.1,0.071,17.0,102.0,0.9978,3.35,0.80,10.5,5,True
16,8.5,0.280,0.56,1.8,0.092,35.0,103.0,0.9969,3.30,0.75,10.5,7,True
31,6.9,0.685,0.00,2.5,0.105,22.0,37.0,0.9966,3.46,0.57,10.6,6,True
36,7.8,0.600,0.14,2.4,0.086,3.0,15.0,0.9975,3.42,0.60,10.8,6,True


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,above_mean_alcohol
9,8.1,0.22,0.43,1.50,0.044,28.0,129.0,0.9938,3.22,0.45,11.0,6,True
10,8.1,0.27,0.41,1.45,0.033,11.0,63.0,0.9908,2.99,0.56,12.0,5,True
12,7.9,0.18,0.37,1.20,0.040,16.0,75.0,0.9920,3.18,0.63,10.8,5,True
13,6.6,0.16,0.40,1.50,0.044,48.0,143.0,0.9912,3.54,0.52,12.4,7,True
15,6.6,0.17,0.38,1.50,0.032,28.0,112.0,0.9914,3.25,0.55,11.4,7,True


---

## Part 3: Binning by Acidity

As a reminder from `02_Combining_DataFrames`, here is how `pd.cut()` bins the red wine data. Your bins for the white wines will have different edges and counts:

![pd.cut splits fixed acidity into 5 equal-width bins, counts how many wines fall in each, and gives each row a bin label](../assets/binning_pd_cut.png)

### Challenge 3: Bin by fixed acidity

You want to avoid wines with the highest acidity. To do this:

1. Use `pd.cut()` to divide the `fixed acidity` column of each DataFrame into **5 equal-width bins**.

2. Attach the resulting bin `Series` to each original DataFrame, once using `.join()` and once using `pd.concat(..., axis=1)`.

3. Filter each DataFrame to return only rows that are **not** in the top acidity bin.

> Note: equal-width bins divide the *range* of values into equal intervals. They are different from quintiles (equal-count bins).

In [8]:
# Create five equal-width fixed-acidity bins for each dataset
red_acidity_bins = pd.cut(red_df["fixed acidity"], bins=5).rename("acidity_bin")
white_acidity_bins = pd.cut(white_df["fixed acidity"], bins=5).rename("acidity_bin")
print("Red acidity bins:", red_acidity_bins.cat.categories)
print("White acidity bins:", white_acidity_bins.cat.categories)

Red acidity bins: IntervalIndex([(4.589, 6.86], (6.86, 9.12], (9.12, 11.38], (11.38, 13.64],
               (13.64, 15.9]],
              dtype='interval[float64, right]')
White acidity bins: IntervalIndex([(3.79, 5.88], (5.88, 7.96], (7.96, 10.04], (10.04, 12.12],
               (12.12, 14.2]],
              dtype='interval[float64, right]')


In [9]:
# Attach acidity bins using .join()
red_acidity_joined = red_df.join(red_acidity_bins)
white_acidity_joined = white_df.join(white_acidity_bins)
red_acidity_joined[["fixed acidity", "acidity_bin"]].head(), white_acidity_joined[["fixed acidity", "acidity_bin"]].head()

(   fixed acidity    acidity_bin
 0            7.4   (6.86, 9.12]
 1            7.8   (6.86, 9.12]
 2            7.8   (6.86, 9.12]
 3           11.2  (9.12, 11.38]
 4            7.4   (6.86, 9.12],
    fixed acidity    acidity_bin
 0            7.0   (5.88, 7.96]
 1            6.3   (5.88, 7.96]
 2            8.1  (7.96, 10.04]
 3            7.2   (5.88, 7.96]
 4            7.2   (5.88, 7.96])

In [10]:
# Attach acidity bins using pd.concat(..., axis=1)
red_acidity_concat = pd.concat([red_df, red_acidity_bins], axis=1)
white_acidity_concat = pd.concat([white_df, white_acidity_bins], axis=1)
red_acidity_concat[["fixed acidity", "acidity_bin"]].head(), white_acidity_concat[["fixed acidity", "acidity_bin"]].head()

(   fixed acidity    acidity_bin
 0            7.4   (6.86, 9.12]
 1            7.8   (6.86, 9.12]
 2            7.8   (6.86, 9.12]
 3           11.2  (9.12, 11.38]
 4            7.4   (6.86, 9.12],
    fixed acidity    acidity_bin
 0            7.0   (5.88, 7.96]
 1            6.3   (5.88, 7.96]
 2            8.1  (7.96, 10.04]
 3            7.2   (5.88, 7.96]
 4            7.2   (5.88, 7.96])

In [11]:
# Exclude each dataset's highest fixed-acidity bin
red_top_acidity_bin = red_acidity_bins.cat.categories[-1]
white_top_acidity_bin = white_acidity_bins.cat.categories[-1]

red_non_top_acidity = red_acidity_joined.loc[
    red_acidity_joined["acidity_bin"] != red_top_acidity_bin
].copy()
white_non_top_acidity = white_acidity_joined.loc[
    white_acidity_joined["acidity_bin"] != white_top_acidity_bin
].copy()

print("Red rows retained:", len(red_non_top_acidity), "of", len(red_df))
print("White rows retained:", len(white_non_top_acidity), "of", len(white_df))

Red rows retained: 1587 of 1599
White rows retained: 4897 of 4898


---

## Part 4: Pivot Tables

As a reminder from `02_Combining_DataFrames`, here is how `pd.pivot_table()` turns rows into a table. Challenge 4 uses only `index`; Challenge 5 adds `columns`, as in the diagram:

![pd.pivot_table turns rows in long format into a table with quality as rows, acidity bins as column headers, and the mean alcohol of each combination in the cells](../assets/pivot_table.png)

### Challenge 4: Alcohol by quality, pivot table

You want to understand how avoiding above-average alcohol affects the quality of wines available to you.

Using the **above-average alcohol** subsets from Challenge 2, create a **pivot table** for each wine type showing:

- The **average `alcohol` content** as the value

- **`quality`** as the index

This should give you one row per quality level and the average alcohol content for that group.

In [12]:
# Average alcohol by quality among red wines above their mean alcohol
red_alcohol_pivot = pd.pivot_table(
    red_above_mean_wines,
    values="alcohol",
    index="quality",
    aggfunc="mean",
)
red_alcohol_pivot

,alcohol
quality,
3,10.866667
4,11.295000
5,11.088686
6,11.437214
7,11.708236
8,12.368750


In [13]:
# Average alcohol by quality among white wines above their mean alcohol
white_alcohol_pivot = pd.pivot_table(
    white_above_mean_wines,
    values="alcohol",
    index="quality",
    aggfunc="mean",
)
white_alcohol_pivot

,alcohol
quality,
3,11.525000
4,11.425532
5,11.211099
6,11.604366
7,11.924761
8,12.139437
9,12.625000


### Challenge 5: Alcohol by quality and acidity bin, pivot table

Now do the same for the **non-top-acidity** subsets from Challenge 3. Create a pivot table for each wine type showing:

- The **average `alcohol` content** as the value

- **`quality`** as the index

- **`fixed acidity` bin** as the columns

This lets you see average alcohol broken down by both quality score and acidity category.

In [14]:
# Average alcohol by quality and fixed-acidity bin for non-top-bin red wines
red_acidity_pivot = pd.pivot_table(
    red_non_top_acidity,
    values="alcohol",
    index="quality",
    columns="acidity_bin",
    aggfunc="mean",
)
red_acidity_pivot

acidity_bin,"(4.589, 6.86]","(6.86, 9.12]","(9.12, 11.38]","(11.38, 13.64]"
quality,,,,
3,9.875000,10.500000,9.150000,9.000000
4,11.006667,9.981667,9.920000,9.966667
5,10.304245,9.773727,9.846429,10.229630
6,11.223529,10.431985,10.687395,10.480952
7,12.195402,11.486413,11.383333,10.809524
8,13.633333,11.937500,11.916667,9.800000


In [15]:
# Average alcohol by quality and fixed-acidity bin for non-top-bin white wines
white_acidity_pivot = pd.pivot_table(
    white_non_top_acidity,
    values="alcohol",
    index="quality",
    columns="acidity_bin",
    aggfunc="mean",
)
white_acidity_pivot

acidity_bin,"(3.79, 5.88]","(5.88, 7.96]","(7.96, 10.04]","(10.04, 12.12]"
quality,,,,
3,9.850000,10.472727,10.540000,9.65
4,10.786364,10.102564,10.126471,9.90
5,10.136635,9.732173,10.176452,9.70
6,10.902765,10.529056,10.635247,9.50
7,12.021405,11.303724,10.988679,NaN
8,12.403030,11.486364,11.080000,NaN
9,NaN,12.625000,10.400000,NaN


---

## Part 5: Open Exploration

### Challenge 6: Your own question

Set a 10-minute timer. Pick one question about the wine data that interests you and answer it using any combination of the tools from this notebook (filtering, joining, binning, pivot tables, groupby, visualisation). Be ready to explain what you found and why you chose that question.

In [17]:
# Exploration question: Does alcohol tend to rise with quality for both wine types?
red_alcohol_by_quality = red_df.groupby("quality")["alcohol"].mean().rename("red_mean_alcohol")
white_alcohol_by_quality = white_df.groupby("quality")["alcohol"].mean().rename("white_mean_alcohol")
quality_alcohol_comparison = pd.concat(
    [red_alcohol_by_quality, white_alcohol_by_quality],
    axis=1,
)
display(quality_alcohol_comparison)

# Spearman correlation is Pearson correlation applied to ranked values
red_quality_alcohol_corr = red_df["quality"].rank().corr(red_df["alcohol"].rank())
white_quality_alcohol_corr = white_df["quality"].rank().corr(white_df["alcohol"].rank())
print(f"Spearman quality-alcohol correlation, red: {red_quality_alcohol_corr:.3f}")
print(f"Spearman quality-alcohol correlation, white: {white_quality_alcohol_corr:.3f}")
print("In both datasets, higher quality is generally associated with higher alcohol content.")

,red_mean_alcohol,white_mean_alcohol
quality,,
3,9.955000,10.345000
4,10.265094,10.152454
5,9.899706,9.808840
6,10.629519,10.575372
7,11.465913,11.367936
8,12.094444,11.636000
9,NaN,12.180000


Spearman quality-alcohol correlation, red: 0.479
Spearman quality-alcohol correlation, white: 0.440
In both datasets, higher quality is generally associated with higher alcohol content.
